Look for any matches with incomplete data.

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
DATA_PROCESSED = ROOT / "data_processed"

in_path = DATA_PROCESSED / "03_tiebreak_pressure_flags.parquet"
df = pd.read_parquet(in_path)

First check if there are any games without 'is_match_end' = True

In [ ]:
match_end_counts = (
    df.groupby("match_id")["is_match_end"]
      .any()         
)
bad_matches = match_end_counts[~match_end_counts].index

print("Matches without match end:", len(bad_matches))

df = df[~df["match_id"].isin(bad_matches)].copy()

print("Remaining matches:", df["match_id"].nunique())
print("Remaining rows:", len(df))

Matches without match end: 0
Remaining matches: 336
Remaining rows: 48211


Check for any retirements.

In [9]:
import numpy as np
import pandas as pd

df = df.sort_values(["match_id","SetNo","GameNo","PointNumber"]).copy()

def add_sets_won_upto(m: pd.DataFrame) -> pd.DataFrame:
    m = m.copy()

    # NA-safe set_end
    m["is_set_end"] = m["is_set_end"].fillna(False).astype(bool)

    # set final games (max within set)
    set_final = (
        m.groupby("SetNo")[["P1GamesWon","P2GamesWon"]]
         .max()
         .sort_index()
    )

    # set winner: 1 / 2 / 0 (0 = unknown/invalid set)
    set_winner = np.where(
        set_final["P1GamesWon"] > set_final["P2GamesWon"], 1,
        np.where(set_final["P2GamesWon"] > set_final["P1GamesWon"], 2, 0)
    )
    set_winner = pd.Series(set_winner, index=set_final.index).astype("int64")

    m["set_winner"] = m["SetNo"].map(set_winner).fillna(0).astype("int64")

    # only count the set once (on is_set_end row)
    p1_set_completed = ((m["set_winner"] == 1) & (m["is_set_end"])).fillna(False).astype("int64")
    p2_set_completed = ((m["set_winner"] == 2) & (m["is_set_end"])).fillna(False).astype("int64")

    m["P1SetsWon_upto"] = p1_set_completed.cumsum().astype("int64")
    m["P2SetsWon_upto"] = p2_set_completed.cumsum().astype("int64")

    return m

df = df.groupby("match_id", group_keys=False).apply(add_sets_won_upto)


/var/folders/sk/00rkx0hd027cbgk6d5rxrl7w0000gn/T/ipykernel_28419/3743578000.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("match_id", group_keys=False).apply(add_sets_won_upto)


In [10]:
last = (
    df.sort_values(["match_id","SetNo","GameNo","PointNumber"])
      .groupby("match_id")
      .tail(1)
)

print(last[["P1SetsWon_upto","P2SetsWon_upto"]].value_counts())


P1SetsWon_upto  P2SetsWon_upto
2               0                 98
0               2                 90
                0                 56
2               1                 49
1               2                 42
                1                  1
Name: count, dtype: int64


In [11]:
# matches where sets never increment by match end
bad_matches = (
    df[df["is_match_end"]]
      .loc[(df["P1SetsWon_upto"] == 0) & (df["P2SetsWon_upto"] == 0), "match_id"]
      .unique()
)
print("Bad matches:", len(bad_matches))

# check if is_set_end ever appears in them
check = (
    df[df["match_id"].isin(bad_matches)]
      .groupby("match_id")["is_set_end"]
      .apply(lambda s: bool(pd.Series(s).fillna(False).any()))
      .value_counts()
)
print(check)

# peek one match
m0 = bad_matches[0]
df[df["match_id"] == m0][["match_id","SetNo","GameNo","PointNumber","is_set_end","is_match_end","P1GamesWon","P2GamesWon"]].tail(30)


Bad matches: 56
is_set_end
True    56
Name: count, dtype: int64


,match_id,SetNo,GameNo,PointNumber,is_set_end,is_match_end,P1GamesWon,P2GamesWon
27334,2018-ausopen-2501,2,13,89,False,False,0,0
27335,2018-ausopen-2501,2,13,90,False,False,0,0
27336,2018-ausopen-2501,2,13,91,False,False,0,0
27337,2018-ausopen-2501,2,13,92,False,False,0,0
27338,2018-ausopen-2501,2,14,93,False,False,0,0
27339,2018-ausopen-2501,2,14,94,False,False,0,0
27340,2018-ausopen-2501,2,14,95,False,False,0,0
27341,2018-ausopen-2501,2,14,96,False,False,0,0
27342,2018-ausopen-2501,2,14,97,False,False,0,0
27343,2018-ausopen-2501,2,15,98,False,False,0,0


In [12]:
completed_matches = (
    df.groupby("match_id")[["P1SetsWon_upto", "P2SetsWon_upto"]]
      .max()
      .query("P1SetsWon_upto == 2 or P2SetsWon_upto == 2")
      .index
)

df = df[df["match_id"].isin(completed_matches)]


Save final dataset

In [13]:
OUT_DIR = DATA_PROCESSED
out_path = OUT_DIR / "04_final_filtered_dataset.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df.shape)

Saved: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_processed/04_final_filtered_dataset.parquet
Shape: (39921, 56)
